# Sistema Ibrido LNN + LLM per Natural Language Understanding

Questo notebook dimostra l'integrazione di **IBM Logical Neural Networks (LNN)** con **Large Language Models (Claude)** per una pipeline completa di NLU.

## Pipeline Ibrida

```
Testo → [LLM] → Fatti strutturati → [LNN] → Inferenze logiche
```

## Cosa Imparerai

- Usare LLM per estrarre fatti da testo non strutturato
- Applicare regole logiche LNN per inferire nuova conoscenza
- **Gestire API keys in modo sicuro** su Colab (best practice 2025)
- Combinare ragionamento simbolico e apprendimento neurale

---

**Nota sull'architettura**: Questo notebook segue le best practice DRY. La logica è nel file `hybrid_nlu.py`.

## Setup Ambiente

In [7]:
# Installazione dipendenze
!pip install -q git+https://github.com/IBM/LNN.git
!pip install -q torch>=2.0.0 numpy>=1.24.0 anthropic>=0.21.0

# Auto-download del modulo per Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("🌐 Esecuzione su Google Colab")
except ImportError:
    IN_COLAB = False
    print("💻 Esecuzione locale")

if IN_COLAB:
    import urllib.request
    url = "https://raw.githubusercontent.com/gianlucamazza/neuro_llm/main/examples/03_hybrid_llm/hybrid_nlu.py"
    print("📥 Download modulo hybrid_nlu.py...")
    urllib.request.urlretrieve(url, "hybrid_nlu.py")
    print("✓ Modulo scaricato correttamente")
else:
    print("✓ Usando file locale hybrid_nlu.py")

print("\n" + "="*50)
print("Setup completato!")
print("="*50)

  Preparing metadata (setup.py) ... done
🌐 Esecuzione su Google Colab
📥 Download modulo hybrid_nlu.py...
✓ Modulo scaricato correttamente

Setup completato!


## Configurazione API Key (Colab Secrets - Best Practice 2025)

### Come configurare la tua API key in modo sicuro:

1. **Clicca sull'icona della chiave** (🔑) nella barra laterale sinistra di Colab
2. **Aggiungi un nuovo secret**:
   - Nome: `ANTHROPIC_API_KEY`
   - Valore: La tua API key Anthropic (inizia con `sk-ant-...`)
3. **Abilita l'accesso** per questo notebook (toggle switch)

### Perché questo metodo è sicuro:

- I secrets sono **crittografati** da Google
- **Non vengono inclusi** quando condividi il notebook
- Ogni utente deve configurare le proprie chiavi
- Nessun hardcoding nel codice

### Se non hai una API key:

Il notebook funzionerà in **DEMO MODE** con fatti predefiniti.

In [13]:
# Carica API key da Colab Secrets (best practice 2025)
try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print("✓ API Key caricata con successo da Colab Secrets")
except Exception as e:
    print("⚠️  ATTENZIONE: API key non trovata nei Colab Secrets")
    print("Il sistema funzionerà in DEMO MODE con fatti predefiniti")
    print("\nPer usare Claude:")
    print("1. Clicca sull'icona chiave (🔑) nella barra laterale")
    print("2. Aggiungi secret 'ANTHROPIC_API_KEY' con la tua chiave")
    print("3. Riavvia questo cell\n")
    ANTHROPIC_API_KEY = None

✓ API Key caricata con successo da Colab Secrets


## Import Moduli

In [14]:
from hybrid_nlu import HybridNLUSystem

print("✓ HybridNLUSystem caricato correttamente")
if ANTHROPIC_API_KEY:
    print("✓ Modalità completa con Claude API")
else:
    print("⚠️  Modalità DEMO (senza Claude API)")

✓ HybridNLUSystem caricato correttamente
✓ Modalità completa con Claude API


## Creazione del Sistema

In [15]:
print("="*70)
print("SISTEMA IBRIDO LNN + LLM per Natural Language Understanding")
print("="*70)

system = HybridNLUSystem(api_key=ANTHROPIC_API_KEY)

if system.has_llm:
    print("\n✓ Sistema creato con Claude API")
else:
    print("\n✓ Sistema creato in DEMO MODE")

SISTEMA IBRIDO LNN + LLM per Natural Language Understanding

✓ Sistema creato con Claude API


## Testo di Input

Forniamo un testo biografico con informazioni su persone e luoghi.

In [16]:
text = """
Leonardo da Vinci nacque a Vinci, un piccolo paese in Toscana, Italia, nel 1452.
Michelangelo Buonarroti nacque a Caprese, sempre in Toscana, nel 1475.
Entrambi vissero gran parte della loro vita a Firenze, che si trova in Toscana.
"""

print("Testo di input:")
print("-" * 70)
print(text)
print("-" * 70)

Testo di input:
----------------------------------------------------------------------

Leonardo da Vinci nacque a Vinci, un piccolo paese in Toscana, Italia, nel 1452.
Michelangelo Buonarroti nacque a Caprese, sempre in Toscana, nel 1475.
Entrambi vissero gran parte della loro vita a Firenze, che si trova in Toscana.

----------------------------------------------------------------------


## Processamento ed Inferenza

Eseguiamo la pipeline passo-passo:
1. **LLM** estrae fatti strutturati dal testo
2. Carichiamo i fatti in **LNN**
3. **LNN** applica regole logiche per inferire nuova conoscenza

In [ ]:
print("[1/4] Estrazione fatti da testo con LLM...")
facts = system.extract_facts_with_llm(text)
print(f"      Estratti: {len(facts['facts'])} fatti, "
      f"{len(facts['entities']['people'])} persone, "
      f"{len(facts['entities']['locations'])} luoghi")

print("\n[2/4] Caricamento fatti in LNN...")
system.add_facts_to_lnn(facts)

print("[3/4] Inferenza logica...")
system.infer()

print("[4/4] Estrazione fatti inferiti...")
inferred_facts = system.extract_inferred_facts()
print(f"      Inferiti: {len(inferred_facts)} nuovi fatti")

print("\n✓ Processamento completato!")

## Risultati: Fatti Inferiti

LNN ha dedotto nuova conoscenza applicando le regole logiche ai fatti estratti dall'LLM.

Ad esempio, se Leonardo nacque a Vinci, LNN inferisce che è cittadino di Vinci (regola: BornIn → CitizenOf).

In [18]:
print("\n" + "="*70)
print("FATTI INFERITI DA LNN:")
print("="*70)

for fact in inferred_facts:
    conf = f"[{fact['confidence'][0]:.2f}, {fact['confidence'][1]:.2f}]"
    print(f"  ✓ {fact['explanation']} (confidenza: {conf})")

print("\n" + "="*70)


FATTI INFERITI DA LNN:


NameError: name 'inferred_facts' is not defined

## Conclusioni

### Pipeline Ibrida Dimostrata:

1. **LLM (Claude)** converte linguaggio naturale → fatti strutturati
2. **LNN** applica ragionamento logico → nuovi fatti inferiti
3. **Gestione sicura API keys** con Colab Secrets (standard 2025)

### Vantaggi dell'Approccio Ibrido:

- **LLM**: Comprensione linguaggio naturale, flessibilità
- **LNN**: Ragionamento logico affidabile, interpretabilità
- **Insieme**: Best of both worlds

### Vs Approcci Alternativi:

| Approccio | Flessibilità | Precisione Logica | Interpretabilità |
|-----------|--------------|-------------------|------------------|
| Solo LLM | Alta | Media | Bassa |
| Solo LNN | Bassa | Alta | Alta |
| **Ibrido** | **Alta** | **Alta** | **Alta** |

### Applicazioni:

- Question answering su documenti
- Knowledge base construction
- Fact checking automatico
- Sistemi esperti conversazionali

### Sicurezza API Keys:

Questo notebook segue le **best practice 2025**:
- ✅ Nessun hardcoding di API keys
- ✅ Uso di Google Colab Secrets Manager
- ✅ Keys crittografate e non condivise
- ✅ Fallback graceful in demo mode